In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,recall_score,confusion_matrix,classification_report

In [3]:
true_df = pd.read_csv("True.csv")
true_df.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [4]:
fake_df = pd.read_csv("Fake.csv")
fake_df.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [5]:
true_df['label'] = 1   # Real news
fake_df['label'] = 0   # Fake news

In [6]:
df = pd.concat([true_df, fake_df], axis=0)
df = df.sample(frac=1).reset_index(drop=True)  # shuffle

In [7]:
from nltk.tokenize import word_tokenize

In [8]:
nltk.download('punkt_tab')
word = word_tokenize("text")
word

[nltk_data] Downloading package punkt_tab to C:\Users\Abdus
[nltk_data]     Soyim\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


['text']

In [9]:
word4= word_tokenize(df['text'].iloc[0])
word5 = list[word4]
print(word5)

list[['A', 'Columbia', 'University', 'professor', 'from', 'Brooklyn', 'went', 'into', 'hiding', 'Thursday', 'after', 'pal', 'James', 'Comey', 'revealed', 'during', 'his', 'Senate', 'testimony', 'that', 'the', 'man', 'leaked', 'memos', 'detailing', 'the', 'former', 'FBI', 'chief', 's', 'conversations', 'with', 'President', 'Trump', 'to', 'the', 'press.But', 'Richman', 'had', 'vanished', 'from', 'his', 'Henry', 'Street', 'digs', 'by', 'midday', ',', 'and', 'family', 'members', ',', 'friends', 'and', 'neighbors', 'wouldn', 't', 'answer', 'doors', 'or', 'phone', 'calls', 'to', 'shed', 'any', 'more', 'light.A', 'doorman', 'eventually', 'turned', 'security', 'guard', 'to', 'stop', 'reporters', 'entering', 'the', 'building.Richman', 's', 'wife', ',', 'Alexandra', 'Bowie', ',', 'is', 'a', 'former', 'president', 'of', 'the', 'influential', 'neighborhood', 'civic', 'group', 'the', 'Brooklyn', 'Heights', 'Association', '.', 'Its', 'director', ',', 'Peter', 'Bray', ',', 'declined', 'to', 'speak', 

In [10]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()
from nltk.stem import LancasterStemmer
ls = LancasterStemmer()
from nltk.stem import SnowballStemmer
sb = SnowballStemmer('english')
from nltk.stem import WordNetLemmatizer
lm = WordNetLemmatizer()

In [11]:
ls.stem("running")

'run'

In [12]:
nltk.download('wordnet')
lm.lemmatize("funding")

[nltk_data] Downloading package wordnet to C:\Users\Abdus
[nltk_data]     Soyim\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


'funding'

In [13]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)      
    text = re.sub(r'[^a-z\s]', '', text)     
    # remove URLs
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return " ".join(words)
df['clean_text'] = df['text'].apply(clean_text)

[nltk_data] Downloading package stopwords to C:\Users\Abdus
[nltk_data]     Soyim\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Abdus
[nltk_data]     Soyim\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [14]:
vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=5000)
X = vectorizer.fit_transform(df['clean_text'])
y = df['label']

In [15]:
vectorizer = TfidfVectorizer(ngram_range=(2,3), max_features=5000)
X = vectorizer.fit_transform(df['clean_text'])
y = df['label']

In [16]:
vectorizer = TfidfVectorizer(ngram_range=(3,4), max_features=5000)
X = vectorizer.fit_transform(df['clean_text'])
y = df['label']

In [17]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [18]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [19]:
y_pred = model.predict(X_test)

In [20]:
scores = cross_val_score(
model, X, y, cv=5, scoring='f1_macro'
)
print("Mean F1:", scores.mean())

Mean F1: 0.9405081604096864


In [25]:
accuracy=accuracy_score(y_test,y_pred)
print("accuracy:",accuracy*100)

accuracy: 94.34298440979956


In [26]:
class_rep=classification_report(y_test,y_pred)
print("class_rep:",class_rep)

class_rep:               precision    recall  f1-score   support

           0       0.93      0.97      0.95      4729
           1       0.97      0.91      0.94      4251

    accuracy                           0.94      8980
   macro avg       0.95      0.94      0.94      8980
weighted avg       0.94      0.94      0.94      8980



In [27]:
confu_ma_result = confusion_matrix(y_test,y_pred)
print("confu_ma:",confu_ma_result)

confu_ma: [[4589  140]
 [ 368 3883]]


In [28]:
def predict_news(text):
    text = clean_text(text)
    vec = vectorizer.transform([text])
    prediction = model.predict(vec)[0]
    return "Real News" if prediction == 1 else "Fake News"
predict_news("Government confirms new economic policy today")

'Fake News'

In [30]:
def predict_news(news_text):
    cleaned = clean_text(news_text)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    probability = model.predict_proba(vectorized)[0].max()

    if prediction == 1:
        return f"✅ Real News (confidence: {probability:.2f})"
    else:
        return f"❌ Fake News (confidence: {probability:.2f})"


news_text = input("Enter News Text: ")
predicted_outcome = predict_news(news_text)

print("Predicted Outcome:", predicted_outcome)


Enter News Text:  Donald Trump Sends Out Embarrassing New Year’.


Predicted Outcome: ❌ Fake News (confidence: 0.75)


In [31]:
import pickle
pickle.dump(model, open(r"E:\Fake News Detection using NLP\model.pkl", "wb"))
pickle.dump(vectorizer, open(r"E:\Fake News Detection using NLP\vectorizer.pkl", "wb"))